In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)


config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)

MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")


OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)

DATA_PATH = os.getenv(
    "DATA_PATH"
)

df=pd.read_csv(f'{DATA_PATH}df_n1.csv')

Using model: gpt-5.6-luna with reasoning effort: low


In [4]:
import ast
import json
import random
import re
import time

# ==========================
# Global token usage
# ==========================

TOTAL_USAGE = {
    "prompt_tokens": 0,
    "reasoning_tokens": 0,
    "total_tokens": 0,
    "num_calls": 0,
}

def update_usage(usage):
    """
    Accumulate token usage across all LLM calls.
    """
    global TOTAL_USAGE

    u = usage.model_dump()

    TOTAL_USAGE["prompt_tokens"] += u.get("prompt_tokens", 0)
    TOTAL_USAGE["total_tokens"] += u.get("total_tokens", 0)
    TOTAL_USAGE["reasoning_tokens"] += (
        u.get("completion_tokens_details", {})
         .get("reasoning_tokens", 0)
    )
    TOTAL_USAGE["num_calls"] += 1
     
     
    
# ==========================
# Helpers: parsing hunk-list fields
# ==========================

def _parse_hunks_field(value):
    """
    pr_code_hunks / same_file_code_hunks are expected to be a list of
    {"filename": ..., "patch": ...} dicts. If the dataframe was loaded
    from CSV they may have come back in as stringified lists, so this
    normalizes both cases to an actual Python list.
    """
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    if isinstance(value, list):
        return value
    return []


# ==========================
# Candidate hunks (the ones the model gets to filter)
# ==========================

def get_same_file_candidates(row):
    """
    Other code hunks touched in the SAME target_file, in the same PR,
    excluding hunk x itself. Matching against `old_hunk`, which holds
    the raw patch text for hunk x as it appears inside
    pr_code_hunks / same_file_code_hunks. Each candidate gets a local id
    (1..N) used to reference it in the model's JSON output.
    """
    hunks = _parse_hunks_field(row.get("same_file_code_hunks"))
    current_patch = row.get("hunk")
    current_patch = str(current_patch).strip() if current_patch is not None else None

    candidates = []
    next_id = 1
    for h in hunks:
        if not isinstance(h, dict):
            continue
        patch = h.get("patch")
        if patch is None:
            continue
        if current_patch is not None and patch.strip() == current_patch:
            continue  # this is hunk x itself, skip it
        candidates.append(
            {
                "id": next_id,
                "filename": h.get("filename", row.get("target_file")),
                "patch": patch,
            }
        )
        next_id += 1
    return candidates


def get_other_file_candidates(row):
    """
    Code hunks touched in OTHER files (not target_file) within the same PR.
    Each candidate gets a local id (1..N), independent from the
    same-file candidate ids.
    """
    hunks = _parse_hunks_field(row.get("pr_code_hunks"))
    target_file = row.get("target_file")

    candidates = []
    next_id = 1
    for h in hunks:
        if not isinstance(h, dict):
            continue
        filename = h.get("filename")
        patch = h.get("patch")
        if filename is None or patch is None:
            continue
        if filename == target_file:
            continue
        candidates.append({"id": next_id, "filename": filename, "patch": patch})
        next_id += 1

    return candidates


def format_candidates(candidates, label):
    """label is 'SF' for same-file candidates or 'OF' for other-file candidates."""
    if not candidates:
        return ""
    blocks = []
    for c in candidates:
        header = f"[{label}-{c['id']}]"
        if label == "OF":
            header += f" file: {c['filename']}"
        # blocks.append(f"{header}\n{_truncate(c['patch'])}")
        blocks.append(f"{header}\n{c['patch']}")
    return "\n\n".join(blocks)


# ==========================
# Prompt construction
# ==========================

def build_relevance_messages(row, same_file_candidates, other_file_candidates):

    system_prompt = """
You are an expert software engineer helping to prepare context for an automated code review comment generator.

You will be given a target code change, called the "target hunk" (<hunk>), that a review comment will be generated for. You will also be given candidate code hunks that were changed elsewhere in the same pull request:

- <same_file_hunks>: other hunks changed in the SAME file as the target hunk.
- <other_file_hunks>: hunks changed in DIFFERENT files, in the same pull request.

Each candidate hunk has a numeric id shown in brackets (e.g. [SF-2], [OF-1]). The two lists use independent numbering — ids restart at 1 in each list. Other-file candidates also show the filename they belong to.

Your task:
For EACH list independently, select ONLY the candidate hunks that are relevant to understanding, evaluating, or reviewing the target hunk. A candidate hunk is relevant if knowing about it would change or inform a reviewer's comment about the target hunk — for example: it touches the same function, class, or variable; it is part of the same logical change (a rename, a signature change, a refactor the target hunk depends on or was caused by); or it establishes behavior/context the target hunk relies on.

A candidate hunk is IRRELEVANT if it is none of the above.

Be selective. The goal is to REDUCE the number of hunks passed downstream to only what a reviewer would actually need to correctly review the target hunk. When a candidate hunk's connection to the target hunk is unclear or weak, exclude it.

Output requirements:
Respond with ONLY a single JSON object and nothing else — don't chnage the code hunks, no explanations, no markdown fences, no extra text. The JSON must have exactly this shape:

{"relevant_same_file_ids": [<ints>], "relevant_other_files_ids": [<ints>]}

Use an empty array for a list if none of its candidates are relevant, or if that list was not shown to you. Never invent ids that were not shown to you.
"""

    user_prompt = f"""
<hunk>
{row["hunk"]}
</hunk>
"""

    same_file_block = format_candidates(same_file_candidates, "SF")
    other_file_block = format_candidates(other_file_candidates, "OF")

    if same_file_block:
        user_prompt += f"""
<same_file_hunks>
{same_file_block}
</same_file_hunks>
"""

    if other_file_block:
        user_prompt += f"""
<other_file_hunks>
{other_file_block}
</other_file_hunks>
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


# ==========================
# Output parsing
# ==========================

def parse_relevance_output(text):
    """
    Expects a JSON object like:
    {"relevant_same_file_ids": [1, 3], "relevant_other_files_ids": [2]}
    Falls back to empty lists on any parsing failure.
    """
    empty = {"relevant_same_file_ids": [], "relevant_other_files_ids": []}

    if text is None:
        return empty

    cleaned = text.strip()
    # strip markdown fences if the model added them despite instructions
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError:
        print("[WARN] Could not parse relevance JSON output")
        return empty

    def _as_int_list(value):
        if not isinstance(value, list):
            return []
        result = []
        for v in value:
            try:
                result.append(int(v))
            except (TypeError, ValueError):
                continue
        return result

    return {
        "relevant_same_file_ids": _as_int_list(data.get("relevant_same_file_ids", [])),
        "relevant_other_files_ids": _as_int_list(data.get("relevant_other_files_ids", [])),
    }


# ==========================
# OpenAI prediction
# ==========================

def predict_relevant_hunks(row):

    same_file_candidates = get_same_file_candidates(row)
    other_file_candidates = get_other_file_candidates(row)

    # Nothing to filter -> skip the LLM call entirely
    if not same_file_candidates and not other_file_candidates:
        return {
            "relevant_same_file_code_hunks": [],
            "relevant_different_files_code_hunks": [],
        }

    messages = build_relevance_messages(row, same_file_candidates, other_file_candidates)

    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "timeout": 120,
    }

    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    response = client.chat.completions.create(**kwargs)
    

    update_usage(response.usage)

    raw_text = response.choices[0].message.content

    parsed = parse_relevance_output(raw_text)

    same_map = {c["id"]: c for c in same_file_candidates}
    other_map = {c["id"]: c for c in other_file_candidates}

    relevant_same_file_code_hunks = [
        {"filename": same_map[i]["filename"], "patch": same_map[i]["patch"]}
        for i in parsed["relevant_same_file_ids"]
        if i in same_map
    ]
    relevant_different_files_code_hunks = [
        {"filename": other_map[i]["filename"], "patch": other_map[i]["patch"]}
        for i in parsed["relevant_other_files_ids"]
        if i in other_map
    ]

    return {
        "relevant_same_file_code_hunks": relevant_same_file_code_hunks,
        "relevant_different_files_code_hunks": relevant_different_files_code_hunks,
    }


# ==========================
# Pipeline execution
# ==========================

def run_relevance_pipeline(df):
    global TOTAL_USAGE

    TOTAL_USAGE = {
    "prompt_tokens": 0,
    "reasoning_tokens": 0,
    "total_tokens": 0,
    "num_calls": 0,
}

    preds = []
    total = len(df)
    processed = 0
    base_sleep = SLEEP_BETWEEN_CALLS

    print(f"[START] Processing {total} rows")

    for _, row in df.iterrows():

        processed += 1
        print(f"\n[ROW {processed}/{total}] Starting")

        if pd.isna(row["hunk"]):
            print("[SKIP] Missing hunk")
            preds.append(
                {
                    "relevant_same_file_code_hunks": [],
                    "relevant_different_files_code_hunks": [],
                }
            )
            continue

        success = False
        sleep_time = base_sleep

        for attempt in range(3):
            try:
                pred = predict_relevant_hunks(row)
                preds.append(pred)
                success = True
                break
            except Exception as e:
                print(f"[ERROR] {e}")
                wait = sleep_time + random.uniform(0, 1)
                print(f"[RETRY] waiting {wait:.2f}s")
                time.sleep(wait)
                sleep_time *= 2

        if not success:
            preds.append(
                {
                    "relevant_same_file_code_hunks": [],
                    "relevant_different_files_code_hunks": [],
                }
            )

        wait = base_sleep + random.uniform(0, 0.8)
        time.sleep(wait)

        if processed % 10 == 0:
            print(f"[CHECKPOINT] processed {processed}/{total}")

    print("\n[DONE] Building dataframe")

    pred_df = pd.DataFrame(preds)
    df = df.reset_index(drop=True)

    return pd.concat([df, pred_df], axis=1)

In [ ]:
df_result=run_relevance_pipeline(df)
df_relevant_hunks=df_result[['patch_id','relevant_different_files_code_hunks','relevant_same_file_code_hunks']]
df_relevant_hunks.to_csv(f'{DATA_PATH}df_n2_2.csv',index=False)

[START] Processing 1 rows

[ROW 1/1] Starting

[DONE] Building dataframe


In [7]:
import ast
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import tiktoken


encoding = tiktoken.get_encoding("o200k_base")

df = df_result.copy()


# ==========================
# Helpers
# ==========================

def count_tokens(text):
    if text is None:
        return 0
    return len(encoding.encode(str(text)))


def serialize_hunks(hunks):
    """
    Convert hunk lists into text for token counting.
    Handles CSV stringified lists.
    """
    if hunks is None:
        return ""

    if isinstance(hunks, float) and pd.isna(hunks):
        return ""

    if isinstance(hunks, str):
        try:
            hunks = ast.literal_eval(hunks)
        except Exception:
            return hunks

    if isinstance(hunks, list):
        return "\n\n".join(
            f"file: {h.get('filename', '')}\n{h.get('patch', '')}"
            if isinstance(h, dict)
            else str(h)
            for h in hunks
        )

    return str(hunks)


def compute_global_stats(df, old_col, new_col):

    original_tokens = 0
    extracted_tokens = 0

    for _, row in df.iterrows():

        original_tokens += count_tokens(
            serialize_hunks(row.get(old_col))
        )

        extracted_tokens += count_tokens(
            serialize_hunks(row.get(new_col))
        )

    compression_ratio = (
        extracted_tokens / original_tokens
        if original_tokens else 0
    )

    return {
        "total_original_tokens": int(original_tokens),
        "total_extracted_tokens": int(extracted_tokens),
        "context_reduction_percentage": round(
            (1 - compression_ratio) * 100,
            2
        ),
        "compression_factor": round(
            original_tokens / extracted_tokens,
            2
        ) if extracted_tokens else 0,
    }


# ==========================
# Compute statistics
# ==========================

same_file_stats = compute_global_stats(
    df,
    "same_file_code_hunks",
    "relevant_same_file_code_hunks",
)


other_file_stats = compute_global_stats(
    df,
    "pr_code_hunks",
    "relevant_different_files_code_hunks",
)


combined_old = 0
combined_new = 0

for _, row in df.iterrows():

    combined_old += count_tokens(
        serialize_hunks(row.get("same_file_code_hunks"))
        + "\n\n"
        + serialize_hunks(row.get("pr_code_hunks"))
    )

    combined_new += count_tokens(
        serialize_hunks(row.get("relevant_same_file_code_hunks"))
        + "\n\n"
        + serialize_hunks(row.get("relevant_different_files_code_hunks"))
    )


combined_stats = {
    "total_original_tokens": int(combined_old),
    "total_extracted_tokens": int(combined_new),
    "context_reduction_percentage": round(
        (1 - combined_new / combined_old) * 100,
        2
    ) if combined_old else 0,
    "compression_factor": round(
        combined_old / combined_new,
        2
    ) if combined_new else 0,
}


# ==========================
# Build log entry
# ==========================

log_entry = {
    "timestamp": datetime.now().isoformat(),
    "task": "relevant_hunk_filtering (2)",
    "model": MODEL_NAME,
    "dataset_length": len(df),

    "same_file_statistics": same_file_stats,
    "other_file_statistics": other_file_stats,
    "combined_statistics": combined_stats,

    "llm_usage": {
        "num_calls": TOTAL_USAGE["num_calls"],
        "prompt_tokens": TOTAL_USAGE["prompt_tokens"],
        "reasoning_tokens": TOTAL_USAGE["reasoning_tokens"],
        "total_tokens": TOTAL_USAGE["total_tokens"],
    }
}


# ==========================
# Save JSON log
# ==========================

log_dir = Path("../logs")
log_dir.mkdir(exist_ok=True)

log_file = log_dir / "token_usage_insights_logs.json"


try:
    with open(log_file, "r", encoding="utf-8") as f:
        logs = json.load(f)

except (FileNotFoundError, json.JSONDecodeError):
    logs = []


logs.append(log_entry)


with open(log_file, "w", encoding="utf-8") as f:
    json.dump(
        logs,
        f,
        indent=4
    )


print(f"Saved statistics to {log_file}")

Saved statistics to ..\logs\token_usage_insights_logs.json
